In [ ]:
exec(open(__import__('pathlib').Path(__vsc_ipynb_file__).parent.parent / 'src' / 'display_html.py').read())

## CHL — Contrastive Hebbian Learning

**Role**: Two-phase, error-driven Hebbian learning without backpropagation (O'Reilly & Munakata 2000 Ch. 4; Schapiro 2017 §2.b).  
Each trial has two phases; the weight update is their difference:

```
ΔW = lr × ( outer(a_pre_plus, a_post_plus) − outer(a_pre_minus, a_post_minus) )
```

- **Minus phase** (ActM): network settles freely → prediction
- **Plus phase** (ActP): output clamped to correct target → correction
- **ΔW = 0** when ActM = ActP (perfect prediction → no update)

**Mask constraint**: sparse connectivity (e.g., DG `ecin_frac=0.25`) means most (pre, post) pairs have no synapse.  
The outer product spans all positions, but `update_weights` zeros out non-existent synapses — sparse structure is preserved throughout training.

**Understanding check**: Why subtract the minus phase rather than applying Hebb on the plus phase alone?  
→ Pure Hebb (ΔW = ActP ⊗ pre) always strengthens co-active pairs → weights grow without bound.  
The minus term subtracts what the network already predicted, so ΔW is proportional to **prediction error**, not raw activity.

In [ ]:
# path & directories
import sys
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DIR_SRC = str((Path(__vsc_ipynb_file__).parent.parent / 'src').resolve())
DIR_VIZ = (Path(__vsc_ipynb_file__).parent.parent / 'visualizations').resolve()
sys.path.insert(0, DIR_SRC)

import torch
from layer import L_DG

# Small DG for visual clarity (realistic: n_input=15, n_DG=100)
N_PRE, N_POST = 8, 20
torch.manual_seed(42)
dg = L_DG(n_input=N_PRE, n_DG=N_POST, k_frac=0.1, ecin_frac=0.5, use_euler=False)

# ECin patterns: moving window (curr=1.0, prev=0.9; Schapiro 2017 §2.c)
a_A = torch.zeros(N_PRE); a_A[0] = 1.0; a_A[2] = 0.9  # item A
a_B = torch.zeros(N_PRE); a_B[2] = 1.0; a_B[4] = 0.9  # item B (shares unit 2)

dg.reset(); act_m = dg(a_A).clone()  # minus phase: item A → DG
dg.reset(); act_p = dg(a_B).clone()  # plus  phase: item B → DG

print(f"Minus: ECin active={torch.where(a_A   > 0)[0].tolist()}, DG winners={torch.where(act_m > 0)[0].tolist()}")
print(f"Plus : ECin active={torch.where(a_B   > 0)[0].tolist()}, DG winners={torch.where(act_p > 0)[0].tolist()}")

## Step 1: Outer product — what positions does CHL touch?

`outer(a_pre, a_post)[i, j] = a_pre[i] × a_post[j]`  
Non-zero only where **both** pre unit i and post unit j are active.  
ΔW is the difference of the two outer products:  
positions **only in plus** → strengthened (red); positions **only in minus** → weakened (blue).

In [ ]:
outer_m = torch.outer(a_A, act_m).numpy()   # (N_PRE, N_POST) — minus contribution
outer_p = torch.outer(a_B, act_p).numpy()   # (N_PRE, N_POST) — plus  contribution
dW_raw  = outer_p - outer_m                 # raw ΔW (before mask applied)

vmax_o = max(float(outer_m.max()), float(outer_p.max()), 1e-6)
vmax_d = max(float(np.abs(dW_raw).max()), 1e-6)

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))

im0 = axes[0].imshow(outer_m.T, cmap='Blues', vmin=0, vmax=vmax_o, aspect='auto')
axes[0].set_title('outer(a_ECin, a_DG) — minus phase\n(subtracted from ΔW)', fontsize=10)
axes[0].set_xlabel('ECin units (pre)')
axes[0].set_ylabel('DG units (post)')
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

im1 = axes[1].imshow(outer_p.T, cmap='Oranges', vmin=0, vmax=vmax_o, aspect='auto')
axes[1].set_title('outer(a_ECin, a_DG) — plus phase\n(added to ΔW)', fontsize=10)
axes[1].set_xlabel('ECin units (pre)')
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

im2 = axes[2].imshow(dW_raw.T, cmap='RdBu_r', vmin=-vmax_d, vmax=vmax_d, aspect='auto')
axes[2].set_title('ΔW = plus − minus\n(red = strengthened, blue = weakened)', fontsize=10)
axes[2].set_xlabel('ECin units (pre)')
plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.savefig(DIR_VIZ / 'CHL_outer_products.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 2: Mask — sparse connectivity preserved after CHL

The outer product in Step 1 assigns a ΔW value to **every** (ECin, DG) pair.  
But DG receives from only `ecin_frac` of ECin units — most pairs have no synapse (mask=0).  
`update_weights` zeros ΔW at mask=0 positions, so non-existent synapses are never created.

In [ ]:
W_before = dg.W.data.clone()
dg.update_weights(a_ECin_minus=a_A, a_ECin_plus=a_B, a_DG_minus=act_m, a_DG_plus=act_p)
dW_actual = (dg.W.data - W_before).numpy()
mask_np   = dg.mask.numpy()                    # (N_PRE, N_POST); 1 = connected

# NaN at mask=0 positions so colormap renders them as gray
dW_masked = np.where(mask_np == 1, dW_actual, np.nan)
cmap_div  = plt.cm.RdBu_r.copy()
cmap_div.set_bad('lightgray')

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))

im0 = axes[0].imshow(dW_raw.T, cmap='RdBu_r', vmin=-vmax_d, vmax=vmax_d, aspect='auto')
axes[0].set_title('ΔW (raw) — outer product spans all positions\nconnected and unconnected alike', fontsize=10)
axes[0].set_xlabel('ECin units (pre)')
axes[0].set_ylabel('DG units (post)')
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

im1 = axes[1].imshow(dW_masked.T, cmap=cmap_div, vmin=-vmax_d, vmax=vmax_d, aspect='auto')
axes[1].set_title(f'ΔW (masked, ecin_frac={dg.mask.mean():.2f}) — gray = no synapse\nnon-existent connections stay exactly zero', fontsize=10)
axes[1].set_xlabel('ECin units (pre)')
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.savefig(DIR_VIZ / 'CHL_mask_constraint.png', dpi=150, bbox_inches='tight')
plt.show()

n_changed   = (torch.tensor(dW_actual)[dg.mask.bool()]  != 0).sum().item()
n_unchanged = (torch.tensor(dW_actual)[~dg.mask.bool()] != 0).sum().item()
print(f"Connected (mask=1) positions changed   : {n_changed}")
print(f"Unconnected (mask=0) positions changed : {n_unchanged}  (must be 0)")